# Vexil Bible Selection Instability 20261113

Independent QA of previously examined Dota 2 development predictions.

This notebook **does not train any model** and is **not a new blind test**.
Source: private frozen dataset, original 291 train/103 validation split.
All 28 new models failed the precommitted selection screen. Original
M1 remains the take-home model.

Numeric source and executable code are pinned by SHA-256.
Not a hosted completion receipt until Kaggle reports COMPLETE.


In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

mode = 'instability'
data = Path(os.environ.get("VEXIL_BIBLE_DATASET", "/kaggle/input/datasets/sangtruong/vexil-bible-development-qa-20261113"))
assert data.is_dir(), f"Attach PRIVATE Kaggle dataset sangtruong/vexil-bible-development-qa-20261113, found {data}"
src = data / "dev_validation_predictions_28.csv"
sha256 = hashlib.sha256(src.read_bytes()).hexdigest()
assert sha256 == 'aebc0cccce1b4c7ed7591bafa3dd2cd52fc1f8de50aa7c79fcba5b2faf3532d4', "Wrong data version; refuse to score"
pack = data / "dataset_files_sha256.json"
assert pack.is_file() and hashlib.sha256(pack.read_bytes()).hexdigest() == 'c9155ee8820fac43f4432ac1f1362226c6c16af7868080b105bd45b30767dc8b', "Dataset manifest drift"
bundle = json.loads(pack.read_text())
for filename, meta in bundle["files"].items():
    item = data / filename
    assert item.is_file(), f"Missing frozen source: {filename}"
    assert hashlib.sha256(item.read_bytes()).hexdigest() == meta["sha256"], f"Source drift: {filename}"
out = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(os.environ.get("VEXIL_BIBLE_OUTPUT", "/tmp/vexil-bible-qa"))
out.mkdir(parents=True,exist_ok=True)
script = data / "verify_bible.py"
subprocess.run([sys.executable,str(script),"--mode",mode,
                "--input-dir",str(data),"--output-dir",str(out)],check=True)
receipt = json.loads((out / f"kaggle_{mode}_receipt.json").read_text())
assert receipt["status"] == "PASS"
assert receipt["source_prediction_sha256"] == sha256
print(json.dumps(receipt, indent=2, sort_keys=True))
